# Complete Lawyer Paper Pipeline

Run the cell below after editing the CONFIG section if needed.

In [1]:
# ============================================================
# LAWYER MASTER + FIRM COUNTS PIPELINE
#
#   Writes
#   1. BrightData_Lawyers_master.csv
#   2. BrightData_Lawyers_master_normalized_1overN.csv
#   3. firm_address_counts_by_MSA.csv
#
# ============================================================

import os
import re
import warnings
from pathlib import Path
import getpass
import numpy as np
import pandas as pd

try:
    import polars as pl
except Exception:
    pl = None

warnings.filterwarnings("ignore")


# ============================================================
# 0. CONFIG
# ============================================================
user = getpass.getuser()

PROJECT_ROOT = Path(f"/Users/{user}/Final_Lawyer_Git July10")
# Main working folder. The three output files will be written here.
OUTPUT_DIR = PROJECT_ROOT / "Data" / "BrightData_Lawyers"

# Where the raw BrightData lawyer CSVs and crosswalk files sit 
DOWNLOADS_DIR = Path.home() / "Downloads"

# Raw BrightData lawyer files (these data are not public)
lawyer_files = [
    DOWNLOADS_DIR / "snap_mi504g7pxmrn977ah.1.csv",
    DOWNLOADS_DIR / "snap_mi504g7pxmrn977ah.2.csv",
    DOWNLOADS_DIR / "snap_mi504g7pxmrn977ah.3.csv",
]

# Geographic crosswalks.
zip_cbsa_path = DOWNLOADS_DIR / "ZIP_CBSA_122024.xlsx"
qcew_crosswalk_path = DOWNLOADS_DIR / "qcew-county-msa-csa-crosswalk-clean.xlsx"

# 13-category practice-area crosswalk.
crosswalk_path = DOWNLOADS_DIR / "brightdata_practice_area_to_12_crosswalk_90pct.csv"

# Keep this True to use the corrected rightmost-ZIP extraction.
USE_CORRECTED_ZIP_EXTRACTION = True

# Puerto Rico metropolitan CBSA codes removed from the MSA list
PR_CBSAS = {41980, 38660, 32420, 25020, 11640, 10380}

# Only these three files are written.
MASTER_OUT = OUTPUT_DIR / "BrightData_Lawyers_master.csv"
NORMALIZED_MASTER_OUT = (
    OUTPUT_DIR / "BrightData_Lawyers_master_normalized_1overN.csv"
)
FIRM_COUNTS_OUT = (
    OUTPUT_DIR / "firm_address_counts_by_MSA.csv"
)

# ============================================================
# 1. HELPERS
# ============================================================

def print_header(title):
    print("\n" + "=" * 60)
    print(title)
    print("=" * 60)


def require_file(path, label):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Missing {label}: {path}")
    return path


def ensure_output_dir():
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def read_csv_robust(path, usecols=None):
    path = str(path)
    if pl is not None:
        try:
            if usecols is not None:
                return pl.read_csv(
                    path,
                    columns=usecols,
                    ignore_errors=True,
                ).to_pandas()
            return pl.read_csv(path, ignore_errors=True).to_pandas()
        except Exception:
            pass
    return pd.read_csv(path, usecols=usecols, low_memory=False)


def clean_area(x):
    """Normalize BrightData practice-area strings."""
    if x is None or pd.isna(x):
        return ""
    x = str(x).strip().lower()
    x = x.replace("[", " ").replace("]", " ")
    x = x.replace('"', " ").replace("'", " ")
    x = x.replace("&amp;", "&")
    x = re.sub(r"\s+", " ", x)
    x = re.sub(r"[^\w\s/&-]", "", x)
    return x.strip()


ZIP_RE = r"(?<!\d)(\d{5})(?:-\d{4})?(?!\d)"


def extract_real_zip_from_text(txt):
    """
    Corrected ZIP extractor: use the rightmost ZIP-like value because an
    address can contain a five-digit street or P.O. box number before its ZIP.
    """
    if not isinstance(txt, str):
        return None
    matches = re.findall(ZIP_RE, txt)
    return matches[-1] if matches else None


def extract_first_zip_from_text(txt):
    """Older first-five-digit behavior retained for compatibility."""
    if not isinstance(txt, str):
        return None
    match = re.search(r"\b\d{5}\b", txt)
    return match.group(0) if match else None


def extract_zip_from_row(row):
    for field in ["mailing_address", "address", "location"]:
        if field in row.index:
            txt = row.get(field)
            if USE_CORRECTED_ZIP_EXTRACTION:
                value = extract_real_zip_from_text(txt)
            else:
                value = extract_first_zip_from_text(txt)
            if value is not None:
                return value
    return None


def make_lawyer_id(df, path):
    """Prefer URL as the stable lawyer ID; otherwise use file path plus row."""
    fallback_ids = pd.Series(
        [f"{path}__{i}" for i in range(len(df))],
        index=df.index,
    )

    if "url" not in df.columns:
        return fallback_ids.astype(str)

    url_ids = df["url"].astype("string").str.strip()
    bad_url = url_ids.isna() | url_ids.str.lower().isin(
        ["", "nan", "none", "null", "na"]
    )
    return url_ids.where(~bad_url, fallback_ids).astype(str)


def check_input_files():
    print_header("CHECKING INPUT FILES")

    for path in lawyer_files:
        require_file(path, "raw BrightData lawyer CSV")
        print("✓", path)

    require_file(zip_cbsa_path, "ZIP-to-CBSA crosswalk")
    print("✓", zip_cbsa_path)

    require_file(qcew_crosswalk_path, "QCEW county-to-MSA crosswalk")
    print("✓", qcew_crosswalk_path)

    if Path(crosswalk_path).exists():
        print("✓", crosswalk_path)
    elif Path(crosswalk_fallback_path).exists():
        print("✓", crosswalk_fallback_path)
    else:
        raise FileNotFoundError(
            "Missing practice-area crosswalk. Check crosswalk_path in CONFIG."
        )


# ============================================================
# 2. REFERENCE TABLES
# ============================================================

def load_practice_area_crosswalk():
    print("\n[1/6] Loading practice-area to 13-category crosswalk...")

    path = Path(crosswalk_path)
    if not path.exists() and Path(crosswalk_fallback_path).exists():
        path = Path(crosswalk_fallback_path)
    require_file(path, "practice-area crosswalk")

    crosswalk = pd.read_csv(path, low_memory=False)
    columns = {column.lower().strip(): column for column in crosswalk.columns}

    key_col = (
        columns.get("practice_area_clean")
        or columns.get("practice_area_key")
        or columns.get("practice_area")
    )
    category_col = (
        columns.get("mapped_12_updated")
        or columns.get("category_12")
        or columns.get("mapped_12")
        or columns.get("category")
    )

    if key_col is None or category_col is None:
        raise ValueError(
            "Could not identify practice-area and category columns in the "
            f"crosswalk. Found: {list(crosswalk.columns)}"
        )

    crosswalk = crosswalk[[key_col, category_col]].copy()
    crosswalk = crosswalk.dropna(subset=[key_col, category_col])
    crosswalk[key_col] = crosswalk[key_col].map(clean_area)
    crosswalk[category_col] = (
        crosswalk[category_col]
        .astype(str)
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

    bad_values = {"", "nan", "none", "null", "na"}
    crosswalk = crosswalk[~crosswalk[key_col].isin(bad_values)]
    crosswalk = crosswalk[
        ~crosswalk[category_col].str.lower().isin(bad_values)
    ]
    crosswalk = crosswalk.drop_duplicates(subset=[key_col], keep="first")

    area_to_category = dict(
        zip(crosswalk[key_col], crosswalk[category_col])
    )
    category_cols = sorted(crosswalk[category_col].unique().tolist())

    print("   Crosswalk path:", path)
    print("   Crosswalk rows:", f"{len(crosswalk):,}")
    print("   Categories:", category_cols)

    if len(category_cols) != 13:
        raise ValueError(
            f"Expected 13 categories, found {len(category_cols)}: {category_cols}"
        )

    return area_to_category, category_cols


def build_msa_reference_tables():
    print("\n[2/6] Building valid MSA and CBSA-name tables...")

    require_file(qcew_crosswalk_path, "QCEW county-to-MSA crosswalk")
    crosswalk = pd.read_excel(qcew_crosswalk_path, dtype=str)
    crosswalk = crosswalk[
        crosswalk["MSA Title"].str.endswith(" MSA", na=False)
    ].copy()

    # Preserve the original pipeline's CBSA conversion logic.
    crosswalk["CBSA"] = (
        crosswalk["MSA Code"]
        .astype(str)
        .str[-4:]
        .str.ljust(5, "0")
        .astype(int)
    )

    valid_msa_codes = set(crosswalk["CBSA"].unique()) - PR_CBSAS

    cbsa_title = (
        crosswalk[["CBSA", "MSA Title"]]
        .drop_duplicates()
        .rename(columns={"MSA Title": "CBSA_Name"})
    )
    cbsa_title = cbsa_title[
        cbsa_title["CBSA"].isin(valid_msa_codes)
    ].copy()
    cbsa_title["CBSA_Name"] = (
        cbsa_title["CBSA_Name"]
        .str.replace(" MSA", "", regex=False)
        .str.replace(
            "Louisville-Jefferson County",
            "Louisville/Jefferson County",
            regex=False,
        )
    )

    print("   Valid MSA codes after Puerto Rico removal:", len(valid_msa_codes))
    print("   MSA names retained:", len(cbsa_title))
    return valid_msa_codes, cbsa_title


def load_zip_cbsa_fixed(valid_msa_codes):
    print("\n[3/6] Loading ZIP-to-CBSA crosswalk...")

    require_file(zip_cbsa_path, "ZIP-to-CBSA crosswalk")
    raw = pd.read_excel(zip_cbsa_path)
    raw["ZIP"] = raw["ZIP"].astype(str).str.zfill(5)
    raw["CBSA"] = pd.to_numeric(raw["CBSA"], errors="coerce")
    raw = raw.dropna(subset=["ZIP", "CBSA"])
    raw["CBSA"] = raw["CBSA"].astype(int)

    raw["is_valid_msa"] = raw["CBSA"].isin(valid_msa_codes).astype(int)
    raw["is_99999"] = (raw["CBSA"] == 99999).astype(int)

    ratio_priority = []
    for column in ["BUS_RATIO", "TOT_RATIO", "RES_RATIO", "OTH_RATIO"]:
        if column in raw.columns:
            raw[column] = pd.to_numeric(
                raw[column], errors="coerce"
            ).fillna(0)
            ratio_priority.append(column)

    sort_cols = ["ZIP", "is_valid_msa", "is_99999"] + ratio_priority + ["CBSA"]
    ascending = [True, False, True] + [False] * len(ratio_priority) + [True]

    zip_to_cbsa = (
        raw.sort_values(sort_cols, ascending=ascending)
        .drop_duplicates(subset="ZIP", keep="first")
        [["ZIP", "CBSA", "is_valid_msa"] + ratio_priority]
        .copy()
    )

    print("   ZIP-to-CBSA rows:", f"{len(zip_to_cbsa):,}")
    print(
        "   ZIPs assigned to valid metro MSA:",
        f"{int(zip_to_cbsa['is_valid_msa'].sum()):,}",
    )
    return zip_to_cbsa


# ============================================================
# 3. LAWYER LABELS AND MSA ASSIGNMENT
# ============================================================

def load_raw_lawyers_with_labels(area_to_category):
    print("\n[4/6] Loading lawyers and mapping practice-area labels...")

    all_lawyer_chunks = []
    label_chunks = []
    raw_row_count = 0

    for path in lawyer_files:
        print("   Loading:", path)
        df = read_csv_robust(path)
        raw_row_count += len(df)

        df["lawyer_id"] = make_lawyer_id(df, path)
        df["zip"] = df.apply(extract_zip_from_row, axis=1)
        all_lawyer_chunks.append(df[["lawyer_id", "zip"]].copy())

        if "areas_of_practice" not in df.columns:
            print("   WARNING: areas_of_practice missing in", path)
            continue

        labels = df[["lawyer_id", "areas_of_practice"]].copy()
        labels["areas_clean"] = (
            labels["areas_of_practice"].astype(str).str.split(",")
        )
        labels = labels[["lawyer_id", "areas_clean"]].explode("areas_clean")
        labels["practice_area_key"] = (
            labels["areas_clean"].astype(str).map(clean_area)
        )
        labels["category_12"] = labels["practice_area_key"].map(
            area_to_category
        )
        labels["category_12"] = labels["category_12"].astype("string")
        labels.loc[
            labels["category_12"].isna()
            | labels["category_12"].str.lower().isin(
                ["", "nan", "none", "null", "na"]
            ),
            "category_12",
        ] = pd.NA

        labels = labels.dropna(subset=["lawyer_id", "category_12"])
        label_chunks.append(labels[["lawyer_id", "category_12"]].copy())

    all_lawyers = (
        pd.concat(all_lawyer_chunks, ignore_index=True)
        .drop_duplicates(subset=["lawyer_id"])
        .reset_index(drop=True)
    )

    if label_chunks:
        labels_long = pd.concat(label_chunks, ignore_index=True)
    else:
        labels_long = pd.DataFrame(columns=["lawyer_id", "category_12"])

    lawyer_category = labels_long.drop_duplicates(
        subset=["lawyer_id", "category_12"]
    )

    print("   Raw rows loaded:", f"{raw_row_count:,}")
    print("   Unique lawyers retained:", f"{all_lawyers['lawyer_id'].nunique():,}")
    print(
        "   Lawyers with at least one mapped label:",
        f"{lawyer_category['lawyer_id'].nunique():,}",
    )
    print("   Unique lawyer-category labels:", f"{len(lawyer_category):,}")

    return all_lawyers, lawyer_category


def build_binary_matrix(all_lawyers, lawyer_category, category_cols):
    if len(lawyer_category) > 0:
        binary = (
            pd.crosstab(
                lawyer_category["lawyer_id"],
                lawyer_category["category_12"],
            )
            .clip(upper=1)
            .astype(int)
            .reset_index()
        )
    else:
        binary = pd.DataFrame({"lawyer_id": []})

    for column in category_cols:
        if column not in binary.columns:
            binary[column] = 0

    binary = binary[["lawyer_id"] + category_cols].copy()

    full_binary = all_lawyers[["lawyer_id"]].merge(
        binary,
        on="lawyer_id",
        how="left",
    )
    for column in category_cols:
        full_binary[column] = full_binary[column].fillna(0).astype(int)

    full_binary["total_labels"] = (
        full_binary[category_cols].sum(axis=1).astype(int)
    )

    print("   Full binary rows:", f"{len(full_binary):,}")
    print(
        "   Total lawyer-category labels:",
        f"{int(full_binary[category_cols].sum().sum()):,}",
    )
    print(
        "   Lawyers with zero mapped labels:",
        f"{int((full_binary['total_labels'] == 0).sum()):,}",
    )
    return full_binary


def assign_msa_to_lawyers(
    all_lawyers,
    zip_to_cbsa,
    valid_msa_codes,
    cbsa_title,
):
    print("\n[5/6] Assigning corrected MSA to each lawyer...")

    lawyer_zip_rows = all_lawyers[["lawyer_id", "zip"]].drop_duplicates(
        subset=["lawyer_id", "zip"]
    )
    lawyer_zip_rows["zip"] = lawyer_zip_rows["zip"].astype("string")

    msa_rows = lawyer_zip_rows.merge(
        zip_to_cbsa,
        left_on="zip",
        right_on="ZIP",
        how="left",
    )
    msa_rows["CBSA"] = pd.to_numeric(msa_rows["CBSA"], errors="coerce")

    msa_rows["msa_status"] = "unknown"
    msa_rows.loc[msa_rows["zip"].isna(), "msa_status"] = "no_zip_extracted"
    msa_rows.loc[
        msa_rows["zip"].notna() & msa_rows["CBSA"].isna(),
        "msa_status",
    ] = "zip_not_in_zip_cbsa_crosswalk"
    msa_rows.loc[
        msa_rows["CBSA"].isin(PR_CBSAS),
        "msa_status",
    ] = "puerto_rico_removed"
    msa_rows.loc[
        msa_rows["CBSA"].notna()
        & ~msa_rows["CBSA"].isin(valid_msa_codes)
        & ~msa_rows["CBSA"].isin(PR_CBSAS),
        "msa_status",
    ] = "cbsa_not_valid_metro_msa"
    msa_rows.loc[
        msa_rows["CBSA"].isin(valid_msa_codes),
        "msa_status",
    ] = "valid_msa"

    print(msa_rows["msa_status"].value_counts(dropna=False).to_string())

    valid_rows = msa_rows[msa_rows["msa_status"] == "valid_msa"].copy()
    if len(valid_rows) > 0:
        valid_rows["CBSA"] = valid_rows["CBSA"].astype(int)
        valid_rows = valid_rows.merge(
            cbsa_title.rename(columns={"CBSA_Name": "MSA"}),
            on="CBSA",
            how="left",
        )

        msa_per_lawyer = (
            valid_rows.groupby(["lawyer_id", "CBSA", "MSA"])
            .size()
            .reset_index(name="n_zip_rows")
            .sort_values(
                ["lawyer_id", "n_zip_rows", "CBSA"],
                ascending=[True, False, True],
            )
            .drop_duplicates(subset="lawyer_id", keep="first")
            [["lawyer_id", "CBSA", "MSA"]]
        )
    else:
        msa_per_lawyer = pd.DataFrame(
            columns=["lawyer_id", "CBSA", "MSA"]
        )

    print(
        "   Lawyers with valid MSA:",
        f"{msa_per_lawyer['lawyer_id'].nunique():,}",
    )
    return msa_per_lawyer


def make_final_lawyer_table(
    full_binary,
    msa_per_lawyer,
    category_cols,
):
    """
    Build the same internal lawyer-level table as the complete pipeline.
    It remains in memory and is not written to disk.
    """
    final = full_binary.merge(
        msa_per_lawyer[["lawyer_id", "CBSA", "MSA"]],
        on="lawyer_id",
        how="left",
    )
    final["MSA"] = final["MSA"].fillna("Unspecified")
    final["CBSA"] = final["CBSA"].astype("Int64")
    final = final.sort_values(["MSA", "lawyer_id"]).reset_index(drop=True)
    final.insert(0, "lawyer_number", np.arange(1, len(final) + 1))

    final_export = final[
        ["lawyer_number"]
        + category_cols
        + ["total_labels", "CBSA", "MSA"]
    ].copy()

    calculated_total_labels = final_export[category_cols].sum(axis=1)
    wrong_total = (
        calculated_total_labels != final_export["total_labels"]
    ).sum()
    if wrong_total != 0:
        raise ValueError(f"Rows with incorrect total_labels: {wrong_total:,}")

    print("   Final lawyer rows:", f"{len(final_export):,}")
    print(
        "   Lawyers with valid MSA:",
        f"{int((final_export['MSA'] != 'Unspecified').sum()):,}",
    )
    print(
        "   Lawyers still Unspecified:",
        f"{int((final_export['MSA'] == 'Unspecified').sum()):,}",
    )
    print(
        "   Unique mapped MSAs:",
        final_export.loc[
            final_export["MSA"] != "Unspecified", "MSA"
        ].nunique(),
    )

    return final, final_export


# ============================================================
# 4. MASTER OUTPUT
# ============================================================

def build_and_save_master(final_export, category_cols):
    """Create the same MSA-level master table used by Lawyer_Git."""
    print("\n[6/6] Building the MSA-level lawyer master...")

    df = final_export.copy()
    denominator = df["total_labels"].replace(0, np.nan)

    normalized_weights = df[category_cols].div(
        denominator,
        axis=0,
    ).fillna(0)
    normalized_weight_cols = [
        f"{category}_weight_1_over_n" for category in category_cols
    ]
    normalized_weights.columns = normalized_weight_cols

    msa_cbsa = df.groupby("MSA")["CBSA"].first().rename("CBSA")
    msa_size = df.groupby("MSA").size().rename("n_lawyers")
    msa_labeled = (
        df.assign(is_labeled=df["total_labels"] > 0)
        .groupby("MSA")["is_labeled"]
        .sum()
        .rename("n_labeled_lawyers")
    )
    msa_zero = (
        df.assign(is_zero_label=df["total_labels"] == 0)
        .groupby("MSA")["is_zero_label"]
        .sum()
        .rename("n_zero_label_lawyers")
    )

    msa_binary_counts = df.groupby("MSA")[category_cols].sum()
    msa_binary_counts.columns = [
        f"{category}_binary_count" for category in category_cols
    ]

    msa_normalized_counts = (
        pd.concat([df[["MSA"]], normalized_weights], axis=1)
        .groupby("MSA")[normalized_weight_cols]
        .sum()
    )
    msa_normalized_counts.columns = category_cols

    msa_normalized_shares = msa_normalized_counts.div(
        msa_normalized_counts.sum(axis=1).replace(0, np.nan),
        axis=0,
    ).fillna(0)

    msa_normalized_counts.columns = [
        f"{category}_normalized_1overN_count"
        for category in category_cols
    ]
    msa_normalized_shares.columns = [
        f"{category}_normalized_1overN_share"
        for category in category_cols
    ]

    master = pd.concat(
        [
            msa_cbsa,
            msa_size,
            msa_labeled,
            msa_zero,
            msa_binary_counts,
            msa_normalized_counts,
            msa_normalized_shares,
        ],
        axis=1,
    ).reset_index()

    master = master[
        ["CBSA", "MSA"]
        + [
            column
            for column in master.columns
            if column not in ["CBSA", "MSA"]
        ]
    ]

    # Dedicated normalized master used by the normalized analysis branch.
    # This matches the output previously created by
    # NormalizationMatrixConstruction.ipynb, but is generated directly from
    # the same in-memory lawyer table to prevent version mismatches.
    normalized_master = pd.concat(
        [
            msa_cbsa,
            msa_size,
            msa_labeled,
            msa_zero,
            msa_normalized_counts,
        ],
        axis=1,
    ).reset_index()

    normalized_master = normalized_master[
        ["CBSA", "MSA", "n_lawyers", "n_labeled_lawyers",
         "n_zero_label_lawyers"]
        + [
            f"{category}_normalized_1overN_count"
            for category in category_cols
        ]
    ]
    normalized_master["CBSA"] = pd.to_numeric(
        normalized_master["CBSA"],
        errors="coerce",
    ).astype("Int64")

    master.to_csv(MASTER_OUT, index=False)
    normalized_master.to_csv(NORMALIZED_MASTER_OUT, index=False)

    print(f"✓ Saved lawyer master: {MASTER_OUT}")
    print(f"✓ Saved normalized lawyer master: {NORMALIZED_MASTER_OUT}")
    print("   Master rows:", f"{len(master):,}")
    print(
        "   Mapped MSA rows:",
        f"{int((master['MSA'] != 'Unspecified').sum()):,}",
    )
    return master, normalized_master


# ============================================================
# 5. FIRM-ADDRESS OUTPUTS
# ============================================================

def choose_address(row):
    for column in ["mailing_address", "address", "location"]:
        if column in row.index:
            value = row.get(column)
            if (
                pd.notna(value)
                and str(value).strip().lower()
                not in ["", "nan", "none", "null", "na"]
            ):
                return str(value)
    return None


def clean_address_for_grouping(value):
    if pd.isna(value):
        return pd.NA
    value = str(value).lower().strip()
    value = re.sub(r"\s+", " ", value)
    value = re.sub(r"[^\w\s#/-]", "", value)
    return value.strip()


def build_and_save_firm_counts(
    zip_to_cbsa,
    msa_per_lawyer,
):
    """
    Build the firm-address counts.
    A firm address is an address shared by at least two unique lawyers.
    """
    print_header("BUILDING FIRM-ADDRESS COUNTS")

    address_chunks = []

    for path in lawyer_files:
        print("Loading addresses from:", path)
        tmp = read_csv_robust(path)
        tmp["lawyer_id"] = make_lawyer_id(tmp, path)
        tmp["zip"] = tmp.apply(extract_zip_from_row, axis=1)
        tmp["address_raw"] = tmp.apply(choose_address, axis=1)
        tmp["address_key"] = tmp["address_raw"].map(
            clean_address_for_grouping
        )
        address_chunks.append(tmp[["lawyer_id", "zip", "address_key"]])

    lawyer_addresses = pd.concat(address_chunks, ignore_index=True)
    lawyer_addresses = lawyer_addresses.dropna(
        subset=["lawyer_id", "zip", "address_key"]
    )
    lawyer_addresses = lawyer_addresses.drop_duplicates(
        subset=["lawyer_id", "zip", "address_key"]
    )

    lawyer_addresses = lawyer_addresses.merge(
        zip_to_cbsa[["ZIP", "CBSA"]],
        left_on="zip",
        right_on="ZIP",
        how="left",
    )
    lawyer_addresses["CBSA"] = pd.to_numeric(
        lawyer_addresses["CBSA"],
        errors="coerce",
    )
    lawyer_addresses = lawyer_addresses.dropna(subset=["CBSA"])
    lawyer_addresses["CBSA"] = lawyer_addresses["CBSA"].astype(int)

    lawyer_addresses = lawyer_addresses.merge(
        msa_per_lawyer[["lawyer_id", "CBSA", "MSA"]],
        on=["lawyer_id", "CBSA"],
        how="inner",
    )

    lawyer_addresses = (
        lawyer_addresses.sort_values(["MSA", "lawyer_id", "address_key"])
        .drop_duplicates(subset=["lawyer_id"], keep="first")
    )


    address_counts = (
        lawyer_addresses.groupby(["CBSA", "MSA", "address_key"])
        .agg(
            lawyers_at_address=("lawyer_id", "nunique"),
        )
        .reset_index()
    )

    address_counts["is_firm_address"] = (
        address_counts["lawyers_at_address"] >= 2
    )
    address_counts["is_solo_address"] = (
        address_counts["lawyers_at_address"] == 1
    )

    firm_address_stats = (
        address_counts.groupby(["CBSA", "MSA"])
        .agg(
            firm_addresses=("is_firm_address", lambda x: int(x.sum())),
            solo_addresses=("is_solo_address", lambda x: int(x.sum())),
            lawyers_in_firms=(
                "lawyers_at_address",
                lambda x: int(x[x >= 2].sum()),
            ),
            lawyers_solo=(
                "lawyers_at_address",
                lambda x: int(x[x == 1].sum()),
            ),
            total_addresses=("address_key", "nunique"),
        )
        .reset_index()
    )

    firm_address_stats["pct_addresses_firms"] = (
        firm_address_stats["firm_addresses"]
        / firm_address_stats["total_addresses"]
    )
    firm_address_stats["pct_addresses_solo"] = (
        firm_address_stats["solo_addresses"]
        / firm_address_stats["total_addresses"]
    )

    firm_address_stats.to_csv(FIRM_COUNTS_OUT, index=False)
    print(f"✓ Saved firm address counts: {FIRM_COUNTS_OUT}")

    return firm_address_stats


# ============================================================
# 6. MAIN RUN
# ============================================================

def main():
    ensure_output_dir()
    print_header("STARTING LAWYER MASTER + FIRM COUNTS PIPELINE")
    print("Output directory:", OUTPUT_DIR)

    check_input_files()

    area_to_category, category_cols = load_practice_area_crosswalk()
    valid_msa_codes, cbsa_title = build_msa_reference_tables()
    zip_to_cbsa = load_zip_cbsa_fixed(valid_msa_codes)

    all_lawyers, lawyer_category = load_raw_lawyers_with_labels(
        area_to_category
    )
    full_binary = build_binary_matrix(
        all_lawyers,
        lawyer_category,
        category_cols,
    )
    msa_per_lawyer = assign_msa_to_lawyers(
        all_lawyers,
        zip_to_cbsa,
        valid_msa_codes,
        cbsa_title,
    )
    final_internal, final_export = make_final_lawyer_table(
        full_binary,
        msa_per_lawyer,
        category_cols,
    )

    master, normalized_master = build_and_save_master(
        final_export,
        category_cols,
    )
    firm_counts = build_and_save_firm_counts(
        zip_to_cbsa,
        msa_per_lawyer,
    )

    print_header("PIPELINE COMPLETE")
    print("Created only these files:")
    print("  ", MASTER_OUT)
    print("  ", NORMALIZED_MASTER_OUT)
    print("  ", FIRM_COUNTS_OUT)

    return master, normalized_master, firm_counts


if __name__ == "__main__":
    master, normalized_master, firm_counts = main()



STARTING LAWYER MASTER + FIRM COUNTS PIPELINE
Output directory: /Users/maxbelykh/Final_Lawyer_Git July10/Data/BrightData_Lawyers

CHECKING INPUT FILES
✓ /Users/maxbelykh/Downloads/snap_mi504g7pxmrn977ah.1.csv
✓ /Users/maxbelykh/Downloads/snap_mi504g7pxmrn977ah.2.csv
✓ /Users/maxbelykh/Downloads/snap_mi504g7pxmrn977ah.3.csv
✓ /Users/maxbelykh/Downloads/ZIP_CBSA_122024.xlsx
✓ /Users/maxbelykh/Downloads/qcew-county-msa-csa-crosswalk-clean.xlsx
✓ /Users/maxbelykh/Downloads/brightdata_practice_area_to_12_crosswalk_90pct.csv

[1/6] Loading practice-area to 13-category crosswalk...
   Crosswalk path: /Users/maxbelykh/Downloads/brightdata_practice_area_to_12_crosswalk_90pct.csv
   Crosswalk rows: 109,282
   Categories: ['Bankruptcy Law', 'Civil Litigation', 'Corporate / Business Law', 'Criminal Defense', 'Employment / Labor Law', 'Estate Planning / Probate / Trusts', 'Family Law', 'General Practice / Other', 'Immigration Law', 'Intellectual Property Law', 'Personal Injury / Torts', 'Real Esta